# Sprint 3 Risk Scoring
This notebook makes improvements from the risk scoring done in Sprint 2. First, there is the addition of weights. This allows for a smart way to count some of the PR metrics more than others based on their relative levels of importance in finding bad actors. Next, there is the ability to add custom risk score thresholds. This gives the ability to change how many providers are classified as high risk. There are also general code improvements to improve reusability and flexibility. <br>
For finding the optimal weighting strategy, quarters 1 and 2 of 2019 were used, with feedback given from HCPF team. The result -- the best weighting strategy -- was then applied to quarters 3-4 of 2019 and quarters 1-2 of 2020. <br>
For finding the optimal threshold, candidate risk score thresholds of 50, 55, 60, 65 are used to see which is best. These are testing on quarters 1 and 2 of 2019 and then applied to quarters 3-4 of 2019 and quarters 1-2 of 2020. <br>

## Contents
0. Imports and Functions
1. Optimal Weights Strategy
2. Optimal Threshold
3. Apply to all quarters of data
4. Save results in Cloud Pak 4 Data for clustering

# 0.0 Imports and Functions
Many of the functions look similar to those of Sprint 2. There are been some minor code improvements.
### 0.1 Import libraries

In [1]:
# import some libraries into the notebook
# all these libraries are pre-installed in Cloud Pak for Data (dont need to download), so all we have to do is import

path = '/project_data/data_asset'

import pandas as pd
pd.set_option('display.max_columns', 500)
import numpy as np
import sys

from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

from collections import Counter

import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline

### 0.2 Functions
These functions include data preprocessing, risk scoring, and bad actor identification

In [8]:
def preprocessor_refined(binary_feats, num_feats):
    '''
    A scikit-learn preprocessor that handles imputation and scaling.
    
    Parameters:
    binary_feats: a python list of the features that are binary
    num_features: a python list of features that are numerical
    
    Returns:
    preprocessor: a scikit-learn object which when called, imputes and scales the data
    feature_names_reordered: a python list of the feature names
    '''
    
    scaler = StandardScaler()
    
    numerical_pipeline = make_column_transformer((make_pipeline(SimpleImputer(strategy='constant', fill_value=0)), binary_feats),
                                                 (make_pipeline(SimpleImputer(strategy='constant', fill_value=0), scaler), num_feats),
                                                 remainder='drop')
    
    preprocessor = make_column_transformer((numerical_pipeline, num_feats), 
                                           remainder='drop')
    
    feature_names_reordered = binary_feats+num_feats
    
    return preprocessor, feature_names_reordered

In [11]:
def apply_data_transform(defined_preprocessor, df, features_names_from_preprocessor, index):
    
    '''
    This function applies the preprocessing function and properly sets indicies so data is ready for scoring.
    
    Parameters:
    defined_preprocessor (scikit-learn object): the preprocessor returned from the preprocessor_refined function
    df (pandas dataframe): data we want to apply the scoring on
    features_names_from_preprocessor (python list): feature names returned from the preprocessor_refined function
    index (pytohn list): the indicies defined outside of this function (provider_id, date, etc)
    
    Returns:
    df_for_scoring (pandas dataframe): the transformed data, ready for scoring
    '''
    
    transformed_q1 = defined_preprocessor.transformers[0][1].fit_transform(df[features_names_from_preprocessor])
    
    # make data a pandas dataframe
    df_refined = pd.DataFrame(data=transformed_q1, columns=features_names_from_preprocessor)
    
    # add the indicies back to the transformed data
    df_intermediate = pd.concat([df[index], df_refined], axis=1)
    
    # set the indicies
    df_for_scoring = df_intermediate.set_index(['PRSCRB_PROV_LOC_ID', 'Max Dispense Date', 'Min Dispense Date'])
    
    df_for_scoring = df_for_scoring.apply(pd.to_numeric)

    
    return df_for_scoring

In [15]:
def scoring_sum(df, weights, threshold=50):
    '''
    This function compute the Risk Score per provider per quarter.
    
    Parameters:
    df (pandas dataframe): transformed and scaled data that is returned in apply_data_transform function
    weights (python dictionary): weights assigned per PR metric
    
    Keyword Args:
    threshold (int): the default threshold risk score for classifying a provider as High Risk
    
    Returns:
    df1 (pandas dataframe): contains the provider's risk score with a bin of risk classification
    
    '''
    
    df = df*weights
    
    df1 = pd.DataFrame()
    
    df1['Sum'] = df.sum(axis=1)
    
    
    # we use infinity as the upper bound of the bin to make sure we include 100 as high risk
    # the min-max scaling is constrained so we wont get a weird result because of this
    mm_scaler = MinMaxScaler(feature_range=(0,100))
    df1['Risk_Score'] = mm_scaler.fit_transform(df1[['Sum']])
    df1['Bin'] = pd.cut(df1['Risk_Score'], [0,40, threshold, float('inf')], labels=['Low Risk','Medium Risk', 'High Risk'], include_lowest=True)
    
    df1 = df1.reset_index()
    
    return df1#[['Sum', 'Risk_Score', 'Bin']]

In [16]:
def bins_for_scoring_groups(df):
    '''
    Extract only the providers who are considered High Risk.
    
    Parameters:
    df (pandas dataframe): df with "Sum", "Risk_Score", and "Bin" columns
    
    Returns:
    bad_actors (python list): provider ID's who are High Risk
    '''
    print(df.groupby(['Bin']).size())
    bad_actors = df.loc[df['Bin']=='High Risk']['PRSCRB_PROV_LOC_ID'].tolist()
    print('List of suspected bad actors: ', bad_actors)
    
    return bad_actors

### 0.3 Lists for data import

In [6]:
# define list of columns before preprocessing
# i can probably add the num_features list and binary_features list here as well. not sure if i should yet?
usecols = ['Max Dispense Date', 'Min Dispense Date', 'PRSCRB_PROV_LOC_ID',
       'PR2', 'PR1', 'PR4', 'PR3', 'PR5', 'PR6', 'PR7', 'PR8',
       'PR9', 'PR10', 'PR11', 'PR13', 'PR15', 'PR18', 'PR19', 'PR21', 'PR22',
       'PR23', 'PR24', 'PR25', 'PR26', 'PR27', 'PR28', 'PR29', 'PR30']
dtype = {'PR15':'category', 'PR18':'category'}

to_drop = ['PR11']
index = ['PRSCRB_PROV_LOC_ID', 'Max Dispense Date', 'Min Dispense Date']
binary_features = ['PR15', 'PR18']

### 0.4 Read in Q1 2019 for weights and threshold

In [7]:
# read in 1 quarter of data to start the scoring
# the '/project_data/data_asset' folder contains the quarterly data

df_q1_2019 = pd.read_csv('/project_data/data_asset/final_op_analysis_01012019_03312019.csv', usecols=usecols, dtype=dtype)
display(df_q1_2019.head())
print(df_q1_2019.shape)

,Max Dispense Date,Min Dispense Date,PRSCRB_PROV_LOC_ID,PR2,PR1,PR4,PR3,PR5,PR6,PR7,PR8,PR9,PR10,PR11,PR13,PR15,PR18,PR19,PR21,PR22,PR23,PR24,PR25,PR26,PR27,PR28,PR29,PR30
0,2019-03-29,2019-01-02,122635,4.176471,302.235294,0.816092,59.057471,0.112426,0.058824,2,0.000000,0.0,0.117647,0.070423,0.40,0,0,0.0,0.117647,3.845711,0.021277,0.0,0.117647,0.0,0.0,0.176471,0,0.0
1,2019-03-31,2019-01-02,137238,3.303030,297.681818,2.449438,220.752809,0.168421,0.015152,9,0.000686,0.0,0.045455,0.238532,0.21,0,0,0.0,0.045455,5.635131,0.000000,0.0,0.181818,0.0,0.0,0.151515,0,0.0
2,2019-03-26,2019-01-01,118005,6.250000,249.250000,0.294118,11.729412,0.017986,0.000000,1,0.000000,0.0,0.500000,0.000000,0.40,0,0,0.0,0.000000,12.411909,0.000000,0.0,0.250000,0.0,0.0,0.250000,0,0.0
3,2019-03-31,2019-01-09,25176,2.571429,253.000000,0.219512,21.597561,0.050847,0.000000,0,0.000000,0.0,0.142857,0.611111,0.28,0,0,0.0,0.000000,17.807810,0.000000,0.0,0.000000,0.0,0.0,0.142857,0,0.0
4,2019-03-29,2019-01-03,114682,2.250000,169.500000,0.104651,7.883721,0.076923,0.250000,0,0.000000,0.0,0.000000,0.000000,0.00,0,0,0.0,0.000000,4.560072,0.000000,0.0,0.000000,0.0,0.0,0.250000,0,0.0


(13451, 28)


In [8]:
# make list for preprocessing
features1 = [c for c in df_q1_2019.columns if c not in index+to_drop]
num_features1 = df_q1_2019[features1].select_dtypes('number').columns.tolist()

In [10]:
# define the data preprocessor
preprocessor_refine1, features_names1 = preprocessor_refined(binary_features, num_features1)

# compute the scoring from the preprocessed data
df_for_scoring1 = apply_data_transform(preprocessor_refine1, df_q1_2019, features_names1, index)
df_for_scoring1.head()

,,,PR15,PR18,PR2,PR1,PR4,PR3,PR5,PR6,PR7,PR8,PR9,PR10,PR13,PR19,PR21,PR22,PR23,PR24,PR25,PR26,PR27,PR28,PR29,PR30
PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,,,,,,,,,,,,,,,,,,,,,,,,
122635,2019-03-29,2019-01-02,0,0,2.790708,2.738961,0.563728,0.969188,-0.203831,0.093473,0.812395,-0.062877,-0.035368,0.825790,1.735514,-0.065166,1.082859,-0.463188,-0.079558,-0.059603,0.498062,-0.076997,-0.017072,-0.132110,0.0,-0.070847
137238,2019-03-31,2019-01-02,0,0,1.828096,2.684930,3.114631,4.800045,0.064605,-0.169611,4.336105,0.428080,-0.035368,0.138295,0.633964,-0.065166,0.263560,-0.415847,-0.215277,-0.059603,0.963222,-0.076997,-0.017072,-0.220751,0.0,-0.070847
118005,2019-03-26,2019-01-01,0,0,5.075930,2.110246,-0.251473,-0.152099,-0.656572,-0.260885,0.309008,-0.062877,-0.035368,4.466964,1.735514,-0.065166,-0.252296,-0.236559,-0.215277,-0.059603,1.457455,-0.076997,-0.017072,0.129066,0.0,-0.070847
25176,2019-03-31,2019-01-09,0,0,1.021803,2.154743,-0.367989,0.081695,-0.499034,-0.260885,-0.194379,-0.062877,-0.035368,1.065867,1.039798,-0.065166,-0.252296,-0.093804,-0.215277,-0.059603,-0.354732,-0.076997,-0.017072,-0.251504,0.0,-0.070847
114682,2019-03-29,2019-01-03,0,0,0.667559,1.163946,-0.547375,-0.243211,-0.374030,1.245137,-0.194379,-0.062877,-0.035368,-0.294572,-0.583539,-0.065166,-0.252296,-0.444289,-0.215277,-0.059603,-0.354732,-0.076997,-0.017072,0.129066,0.0,-0.070847


### 0.5 Read in Q2 2019 for weights and threshold

In [11]:
# read in 1 quarter of data to start the scoring
# the '/project_data/data_asset' folder contains the quarterly data

df_q2_2019 = pd.read_csv('/project_data/data_asset/final_op_analysis_04012019_06302019.csv', usecols=usecols, dtype=dtype)
display(df_q2_2019.head())
print(df_q2_2019.shape)

,Max Dispense Date,Min Dispense Date,PRSCRB_PROV_LOC_ID,PR2,PR1,PR4,PR3,PR5,PR6,PR7,PR8,PR9,PR10,PR11,PR13,PR15,PR18,PR19,PR21,PR22,PR23,PR24,PR25,PR26,PR27,PR28,PR29,PR30
0,2019-06-30,2019-04-01,105156,2.484848,160.621212,1.802198,116.494505,0.063004,0.045455,5,0.0,0.0,0.060606,0.280488,0.22,0,0,0.0,0.060606,4.597287,0.0,0.0,0.060606,0.0,0.0,0.227273,0.0,0.0
1,2019-06-12,2019-04-29,105157,1.333333,18.333333,0.088889,1.222222,0.015873,0.000000,0,0.0,0.0,0.333333,1.000000,0.00,0,0,0.0,0.000000,2.849320,0.0,0.0,0.000000,0.0,0.0,0.333333,0.0,0.0
2,2019-06-27,2019-04-05,105159,1.846154,108.307692,0.285714,16.761905,0.134615,0.000000,0,0.0,0.0,0.000000,0.125000,0.25,0,0,0.0,0.000000,4.137621,0.0,0.0,0.000000,0.0,0.0,0.153846,0.0,0.0
3,2019-06-19,2019-04-17,105160,1.200000,36.000000,0.093750,2.812500,0.000000,0.000000,0,0.0,0.0,0.200000,1.000000,0.00,0,1,0.0,0.000000,5.601543,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,0.0
4,2019-06-27,2019-06-27,105162,1.000000,30.000000,1.000000,30.000000,0.000000,0.000000,0,0.0,0.0,0.000000,0.000000,0.00,0,0,0.0,0.000000,4.480850,0.0,0.0,0.000000,0.0,0.0,1.000000,0.0,0.0


(13449, 28)


In [12]:
# make list for preprocessing
features = [c for c in df_q2_2019.columns if c not in index+to_drop]
num_features2 = df_q2_2019[features].select_dtypes('number').columns.tolist()

In [13]:
# define the data preprocessor
preprocessor_refine, features_names = preprocessor_refined(binary_features, num_features2)

# compute the scoring from the preprocessed data
df_for_scoring2 = apply_data_transform(preprocessor_refine, df_q2_2019, features_names, index)
df_for_scoring2.head()

,,,PR15,PR18,PR2,PR1,PR4,PR3,PR5,PR6,PR7,PR8,PR9,PR10,PR13,PR19,PR21,PR22,PR23,PR24,PR25,PR26,PR27,PR28,PR29,PR30
PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,,,,,,,,,,,,,,,,,,,,,,,,
105156,2019-06-30,2019-04-01,0,0,0.911584,1.041158,2.242780,2.424984,-0.445382,0.001336,2.248811,-0.116609,-0.038766,0.326983,0.732193,-0.046334,0.439779,-0.455077,-0.22311,-0.0764,0.080425,-0.054871,-0.014417,0.039345,-0.081178,-0.076226
105157,2019-06-12,2019-04-29,0,0,-0.333724,-0.615849,-0.579224,-0.395831,-0.670790,-0.267375,-0.184982,-0.116609,-0.038766,3.067836,-0.574684,-0.046334,-0.244292,-0.500994,-0.22311,-0.0764,-0.358649,-0.054871,-0.014417,0.413090,-0.081178,-0.076226
105159,2019-06-27,2019-04-05,0,0,0.220867,0.431943,-0.255031,-0.015561,-0.102891,-0.267375,-0.184982,-0.116609,-0.038766,-0.282095,0.910403,-0.046334,-0.244292,-0.467152,-0.22311,-0.0764,-0.358649,-0.054871,-0.014417,-0.219401,-0.081178,-0.076226
105160,2019-06-19,2019-04-17,0,1,-0.477917,-0.410113,-0.571217,-0.356915,-0.746704,-0.267375,-0.184982,-0.116609,-0.038766,1.727863,-0.574684,-0.046334,-0.244292,-0.428697,-0.22311,-0.0764,-0.358649,-0.054871,-0.014417,-0.761536,-0.081178,-0.076226
105162,2019-06-27,2019-06-27,0,0,-0.694208,-0.479986,0.921474,0.308387,-0.746704,-0.267375,-0.184982,-0.116609,-0.038766,-0.282095,-0.574684,-0.046334,-0.244292,-0.458136,-0.22311,-0.0764,-0.358649,-0.054871,-0.014417,2.762343,-0.081178,-0.076226


# 1.0 Optimal Weights Strategy
There are 4 main methods for finding the best weights for the PR metrics. The first method is all PR metrics have the same weight (similar to what was done in Sprint 2).<br>
The second method is using the weights from the priority list provided by Jeff M. which can be found in the shared Box folder [here](https://ibm.box.com/s/kaikr55m0wwdoguvindcs7751yvi1md7). <br>
The third method is similar to second, the main difference is adding very high weights for the binary PR metrics, PR15 and PR 18. The motivation for using large weights for the binary variables is these events signify very suspicious behavior. <br>
The fourth method requires using the result of Sprint 2 (Section 1.1.2 Find driving PR's in Sprint2_TG notebook). The weights will be the order of PR importance. So the most important PR metrics gets the highest weight, the second most important gets the second highest weight, and so on. <br>
These methods will be compared with eachother to see which best identifies bad actors of Q1 2019 and Q2 2019. The best weighting strategy will then be applied to the rest of the data.
## 1.1 Make Weights

### 1.1.1 Equal Weights for all PR metrics

In [14]:
# make a dictionary for weights

scoring_columns = []
for col in df_for_scoring1.columns:
    scoring_columns.append(col)
    
#columns
dictionary_of_weights_1 = dict.fromkeys(scoring_columns,1)
dictionary_of_weights_1

{'PR15': 1,
 'PR18': 1,
 'PR2': 1,
 'PR1': 1,
 'PR4': 1,
 'PR3': 1,
 'PR5': 1,
 'PR6': 1,
 'PR7': 1,
 'PR8': 1,
 'PR9': 1,
 'PR10': 1,
 'PR13': 1,
 'PR19': 1,
 'PR21': 1,
 'PR22': 1,
 'PR23': 1,
 'PR24': 1,
 'PR25': 1,
 'PR26': 1,
 'PR27': 1,
 'PR28': 1,
 'PR29': 1,
 'PR30': 1}

### 1.1.2 Weights from Priority List

In [15]:
## if we want to change the weights so some metrics count more than others, we can edit the dictionary directly using the example below

# dictionary_of_weights['PR10'] = 2   # choose the metrics we want to change ('PR10') and the value we want to assign (2)
# dictionary_of_weights               # print the updated dictionary to see the change

dictionary_of_weights_2 = {'PR1':5,
                        'PR2':5,
                        'PR3':5,
                        'PR4':5,
                        'PR5':4,
                        'PR6':4,
                        'PR7':5,
                        'PR8':5,
                        'PR9':5,
                        'PR10':4,
                     #   'PR11':3,
                        'PR13':3,
                        'PR15':4,
                        'PR18':4,
                        'PR19':2,
                        "PR21":2,
                        'PR22':4,
                        'PR23':3,
                        'PR24':2,
                        'PR25':2,
                        'PR26':2,
                        'PR27':2,
                        'PR28':2,
                        'PR29':4,
                        'PR30':4}
dictionary_of_weights_2

{'PR1': 5,
 'PR2': 5,
 'PR3': 5,
 'PR4': 5,
 'PR5': 4,
 'PR6': 4,
 'PR7': 5,
 'PR8': 5,
 'PR9': 5,
 'PR10': 4,
 'PR13': 3,
 'PR15': 4,
 'PR18': 4,
 'PR19': 2,
 'PR21': 2,
 'PR22': 4,
 'PR23': 3,
 'PR24': 2,
 'PR25': 2,
 'PR26': 2,
 'PR27': 2,
 'PR28': 2,
 'PR29': 4,
 'PR30': 4}

### 1.1.3 Weights from Priority List with Heavy Binary Weighting

In [16]:
## if we want to change the weights so some metrics count more than others, we can edit the dictionary directly using the example below


dictionary_of_weights_3 = {'PR1':5,
                        'PR2':5,
                        'PR3':5,
                        'PR4':5,
                        'PR5':4,
                        'PR6':4,
                        'PR7':5,
                        'PR8':5,
                        'PR9':5,
                        'PR10':4,
                     #   'PR11':3,
                        'PR13':3,
                        'PR15':20,
                        'PR18':20,
                        'PR19':2,
                        "PR21":2,
                        'PR22':4,
                        'PR23':3,
                        'PR24':2,
                        'PR25':2,
                        'PR26':2,
                        'PR27':2,
                        'PR28':2,
                        'PR29':4,
                        'PR30':4}
dictionary_of_weights_3

{'PR1': 5,
 'PR2': 5,
 'PR3': 5,
 'PR4': 5,
 'PR5': 4,
 'PR6': 4,
 'PR7': 5,
 'PR8': 5,
 'PR9': 5,
 'PR10': 4,
 'PR13': 3,
 'PR15': 20,
 'PR18': 20,
 'PR19': 2,
 'PR21': 2,
 'PR22': 4,
 'PR23': 3,
 'PR24': 2,
 'PR25': 2,
 'PR26': 2,
 'PR27': 2,
 'PR28': 2,
 'PR29': 4,
 'PR30': 4}

### 1.1.4 Weights from Sort Order Feature Importance

In [17]:
## if we want to change the weights so some metrics count more than others, we can edit the dictionary directly using the example below

# dictionary_of_weights['PR10'] = 2   # choose the metrics we want to change ('PR10') and the value we want to assign (2)
# dictionary_of_weights               # print the updated dictionary to see the change

dictionary_of_weights_4 = {'PR1':16,
                        'PR2':15,
                        'PR3':22,
                        'PR4':20,
                        'PR5':13,
                        'PR6':6,
                        'PR7':24,
                        'PR8':18,
                        'PR9':19,
                        'PR10':9,
                     #   'PR11':3,
                        'PR13':12,
                        'PR15':1,
                        'PR18':2,
                        'PR19':14,
                        "PR21":8,
                        'PR22':21,
                        'PR23':5,
                        'PR24':17,
                        'PR25':11,
                        'PR26':4,
                        'PR27':23,
                        'PR28':10,
                        'PR29':3,
                        'PR30':7}
dictionary_of_weights_4

{'PR1': 16,
 'PR2': 15,
 'PR3': 22,
 'PR4': 20,
 'PR5': 13,
 'PR6': 6,
 'PR7': 24,
 'PR8': 18,
 'PR9': 19,
 'PR10': 9,
 'PR13': 12,
 'PR15': 1,
 'PR18': 2,
 'PR19': 14,
 'PR21': 8,
 'PR22': 21,
 'PR23': 5,
 'PR24': 17,
 'PR25': 11,
 'PR26': 4,
 'PR27': 23,
 'PR28': 10,
 'PR29': 3,
 'PR30': 7}

## 1.2 Apply Weights
The weights are applied using the `scoring_sum` function call. The output is the number of providers determined as High Risk and the list of Provider_ID's.

### 1.2.1 Equal Weights Q1 2019

In [18]:
# USING PRIMITIVE WEIGHTS

#Scale the scoring
df_q1_2019_weights1 = scoring_sum(df_for_scoring1, dictionary_of_weights_1)

# output the bins with sizes
q1_2019_weights1 = bins_for_scoring_groups(df_q1_2019_weights1)

Bin
Low Risk       13413
Medium Risk       23
High Risk         15
dtype: int64
List of suspected bad actors:  [165027, 122949, 156800, 107410, 111356, 152853, 139433, 3241, 16638, 17522, 121269, 111598, 112675, 149181, 166296]


### 1.2.2 Priority Level as Weights Q1 2019
- the weights are the level of priority given in the shared box folder

In [19]:
# using priority levels

#Scale the scoring
df_q1_2019_weights2 = scoring_sum(df_for_scoring1, dictionary_of_weights_2)

# output the bins with sizes
q1_2019_weights2 = bins_for_scoring_groups(df_q1_2019_weights2)

Bin
Low Risk       13428
Medium Risk       14
High Risk          9
dtype: int64
List of suspected bad actors:  [165027, 156800, 107410, 152853, 139433, 16638, 17522, 112675, 166296]


### 1.2.3 Priority Level with Heavy Weights for Binary Variables Q1 2019

In [20]:
# using high binary weights

#Scale the scoring
df_q1_2019_weights3 = scoring_sum(df_for_scoring1, dictionary_of_weights_3)

# output the bins with sizes
q1_2019_weights3 = bins_for_scoring_groups(df_q1_2019_weights3)

Bin
Low Risk       13428
Medium Risk       14
High Risk          9
dtype: int64
List of suspected bad actors:  [165027, 156800, 107410, 152853, 139433, 16638, 17522, 112675, 166296]


### Notes
- Through 3 rounds, we are identifying basically the same providers regardless of weights
- in V1, we dont identify anyone with PR18 = 1 
- in this work, we dont identify them either...lets look at their risk scores then and now to see if they are getting missed totally or what the deal is
    - their risk scores in sprint 2 were pretty low
    - their risk scores in sprint 3 are also pretty low

### 1.2.4 Weights from Sort Order Feature Importance Q1 2019

In [21]:
# using sort order weights

#Scale the scoring
df_q1_2019_weights4 = scoring_sum(df_for_scoring1, dictionary_of_weights_4, threshold=50)

# output the bins with sizes
q1_2019_weights4 = bins_for_scoring_groups(df_q1_2019_weights4)

Bin
Low Risk       13426
Medium Risk       14
High Risk         11
dtype: int64
List of suspected bad actors:  [165027, 156800, 107410, 152853, 139433, 16638, 17522, 121269, 111598, 112675, 149181]


In [22]:
# see how many providers are identified in all weighting schemes

print('Providers identified in all rounds: ', len(list(set(q1_2019_weights1)&set(q1_2019_weights2)&set(q1_2019_weights3)&set(q1_2019_weights4))))

Providers identified in all rounds:  8


### 1.2.5 Equal Weights Q2 2019

In [23]:
# USING PRIMITIVE WEIGHTS

#Scale the scoring
df_q2_2019_weights1 = scoring_sum(df_for_scoring2, dictionary_of_weights_1)

# output the bins with sizes
q2_2019_weights1 = bins_for_scoring_groups(df_q2_2019_weights1)

Bin
Low Risk       13414
Medium Risk       17
High Risk         18
dtype: int64
List of suspected bad actors:  [111813, 107410, 110160, 122907, 136494, 139433, 135733, 153504, 156800, 164110, 165027, 149990, 166304, 16638, 167565, 170861, 17522, 2755]


### 1.2.6 Priority Level as Weights Q2 2019
- the weights are the level of priority given in the shared box folder

In [24]:
# using priority levels

#Scale the scoring
df_q2_2019_weights2 = scoring_sum(df_for_scoring2, dictionary_of_weights_2)

# output the bins with sizes
q2_2019_weights2 = bins_for_scoring_groups(df_q2_2019_weights2)

Bin
Low Risk       13425
Medium Risk       14
High Risk         10
dtype: int64
List of suspected bad actors:  [111813, 107410, 122907, 139433, 156800, 164110, 165027, 149990, 16638, 17522]


### 1.2.7 Priority Level with Heavy Weights for Binary Variables Q2 2019

In [25]:
# using binary weights

#Scale the scoring
df_q2_2019_weights3 = scoring_sum(df_for_scoring2, dictionary_of_weights_3)

# output the bins with sizes
q2_2019_weights3 = bins_for_scoring_groups(df_q2_2019_weights3)

Bin
Low Risk       13425
Medium Risk       14
High Risk         10
dtype: int64
List of suspected bad actors:  [111813, 107410, 122907, 139433, 156800, 164110, 165027, 149990, 16638, 17522]


### 1.2.8 Weights from Sort Order Feature Importance Q2 2019

In [26]:
# using sort order weights

#Scale the scoring
df_q2_2019_weights4 = scoring_sum(df_for_scoring2, dictionary_of_weights_4, threshold=50)

# output the bins with sizes
q2_2019_weights4 = bins_for_scoring_groups(df_q2_2019_weights4)

Bin
Low Risk       13426
Medium Risk       13
High Risk         10
dtype: int64
List of suspected bad actors:  [111813, 107410, 110160, 139433, 156800, 165027, 149990, 16638, 17522, 2755]


### 1.2.9 Get the Y/N indicator from HCPF
This will help us to determine which of the weighting systems best identifies past bad actors

In [27]:
# jeff M gave us indicators of suspicious providers
# use this to determine optimal weighting scheme

y_ind = [139433, 16638, 122949, 149990, 153504, 107410, 17522, 4457, 117764, 61934]
n_ind = [152853, 136494, 110160, 2755, 122907, 166296, 149181, 112675, 111813, 3241, 
         135733, 156800, 164110, 166304, 165027, 121269, 111356, 170861, 111598, 167565,
         153791, 120451, 126953, 148485, 155049, 165594, 131365, 175614, 101951, 170094, 117265, 105543, 157335]

## 1.3 Check which of the weighting systems best identifies past bad actors
The criteria used for determining the best weights is: 
- look at how many of the IND='Y' are in the High Risk Bin
- look at ratio of High Risk are IND='Y"
- look at mean risk Score of IND='N" are high risk (if they have a mean close to 50, we can set higher threshold for better result)

### 1.3.1 Weighting System 1 Q1 2019
Does an OK job of correctly identifying past bad actors. Probably classifies too many 'N' as 'Y'

In [28]:
# for q1 2019, add the indicator column and display 
df_q1_2019_weights1['IND'] = np.where(df_q1_2019_weights1['PRSCRB_PROV_LOC_ID'].isin(y_ind), 'Y', 'N')
df_q1_2019_weights1.loc[df_q1_2019_weights1['IND']=='Y']

,PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,Sum,Risk_Score,Bin,IND
32,122949,2019-03-31,2019-01-01,58.305591,52.915751,High Risk,Y
54,107410,2019-03-31,2019-01-01,85.430544,74.859546,High Risk,Y
3294,139433,2019-03-31,2019-01-01,71.693353,63.746306,High Risk,Y
5051,16638,2019-03-31,2019-01-02,116.506921,100.000000,High Risk,Y
5065,17522,2019-03-30,2019-01-01,75.528314,66.848748,High Risk,Y
6751,153504,2019-03-31,2019-01-01,49.794135,46.030075,Medium Risk,Y
6803,4457,2019-03-31,2019-01-01,51.318056,47.262910,Medium Risk,Y
10128,149990,2019-03-31,2019-01-01,52.234858,48.004593,Medium Risk,Y


In [29]:
print('Value counts per indicator\n', df_q1_2019_weights1.loc[df_q1_2019_weights1['Bin']=='High Risk']
                                                      .sort_values(by='Risk_Score', ascending=False)['IND'].value_counts())

print('\nMean score per indicator', df_q1_2019_weights1.loc[df_q1_2019_weights1['Bin']=='High Risk']
                                                      .groupby(['IND'])['Risk_Score'].mean())



Value counts per indicator
 N    10
Y     5
Name: IND, dtype: int64

Mean score per indicator IND
N    62.439534
Y    71.674070
Name: Risk_Score, dtype: float64


### 1.3.2 Weighting System 2 Q1 2019
Does not do a very good job of correctly identifying past bad actors. Too many 'Y' are in Medium Risk bin.

In [30]:
df_q1_2019_weights2['IND'] = np.where(df_q1_2019_weights2['PRSCRB_PROV_LOC_ID'].isin(y_ind), 'Y', 'N')
df_q1_2019_weights2.loc[df_q1_2019_weights2['IND']=='Y']

,PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,Sum,Risk_Score,Bin,IND
32,122949,2019-03-31,2019-01-01,270.408178,49.834326,Medium Risk,Y
54,107410,2019-03-31,2019-01-01,316.794435,57.598638,High Risk,Y
3294,139433,2019-03-31,2019-01-01,343.464876,62.062840,High Risk,Y
5051,16638,2019-03-31,2019-01-02,570.112497,100.000000,High Risk,Y
5065,17522,2019-03-30,2019-01-01,360.502336,64.914637,High Risk,Y
6751,153504,2019-03-31,2019-01-01,236.548729,44.166800,Medium Risk,Y
6803,4457,2019-03-31,2019-01-01,249.233476,46.290023,Medium Risk,Y
10128,149990,2019-03-31,2019-01-01,239.308794,44.628791,Medium Risk,Y


In [31]:
df_q1_2019_weights2.loc[df_q1_2019_weights2['Bin']=='High Risk'].sort_values(by='Risk_Score', ascending=False)

,PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,Sum,Risk_Score,Bin,IND
5051,16638,2019-03-31,2019-01-02,570.112497,100.000000,High Risk,Y
1602,152853,2019-01-07,2019-01-07,541.972161,95.289761,High Risk,N
5065,17522,2019-03-30,2019-01-01,360.502336,64.914637,High Risk,Y
3294,139433,2019-03-31,2019-01-01,343.464876,62.062840,High Risk,Y
54,107410,2019-03-31,2019-01-01,316.794435,57.598638,High Risk,Y
12082,166296,2019-03-04,2019-03-04,314.235782,57.170361,High Risk,N
7970,112675,2019-02-05,2019-02-05,311.479700,56.709038,High Risk,N
50,156800,2019-03-31,2019-01-01,296.936766,54.274785,High Risk,N
6,165027,2019-03-31,2019-01-01,287.699760,52.728659,High Risk,N


In [32]:
print('Value counts per indicator\n', df_q1_2019_weights2.loc[df_q1_2019_weights2['Bin']=='High Risk']
                                              .sort_values(by='Risk_Score', ascending=False)['IND'].value_counts())
print('\nMean per indicator', df_q1_2019_weights2.loc[df_q1_2019_weights2['Bin']=='High Risk']
                                              .groupby(['IND'])['Risk_Score'].mean())



Value counts per indicator
 N    5
Y    4
Name: IND, dtype: int64

Mean per indicator IND
N    63.234521
Y    71.144029
Name: Risk_Score, dtype: float64


### 1.3.3 Weighting System 3 Q1 2019
Does not do a very good job of correctly identifying past bad actors. Too many 'Y' are in Medium Risk bin.

In [33]:
df_q1_2019_weights3['IND'] = np.where(df_q1_2019_weights3['PRSCRB_PROV_LOC_ID'].isin(y_ind), 'Y', 'N')
df_q1_2019_weights3.loc[df_q1_2019_weights3['IND']=='Y']

,PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,Sum,Risk_Score,Bin,IND
32,122949,2019-03-31,2019-01-01,270.408178,49.834326,Medium Risk,Y
54,107410,2019-03-31,2019-01-01,316.794435,57.598638,High Risk,Y
3294,139433,2019-03-31,2019-01-01,343.464876,62.062840,High Risk,Y
5051,16638,2019-03-31,2019-01-02,570.112497,100.000000,High Risk,Y
5065,17522,2019-03-30,2019-01-01,360.502336,64.914637,High Risk,Y
6751,153504,2019-03-31,2019-01-01,236.548729,44.166800,Medium Risk,Y
6803,4457,2019-03-31,2019-01-01,249.233476,46.290023,Medium Risk,Y
10128,149990,2019-03-31,2019-01-01,239.308794,44.628791,Medium Risk,Y


In [34]:
df_q1_2019_weights3.loc[df_q1_2019_weights3['Bin']=='High Risk'].sort_values(by='Risk_Score', ascending=False)

,PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,Sum,Risk_Score,Bin,IND
5051,16638,2019-03-31,2019-01-02,570.112497,100.000000,High Risk,Y
1602,152853,2019-01-07,2019-01-07,541.972161,95.289761,High Risk,N
5065,17522,2019-03-30,2019-01-01,360.502336,64.914637,High Risk,Y
3294,139433,2019-03-31,2019-01-01,343.464876,62.062840,High Risk,Y
54,107410,2019-03-31,2019-01-01,316.794435,57.598638,High Risk,Y
12082,166296,2019-03-04,2019-03-04,314.235782,57.170361,High Risk,N
7970,112675,2019-02-05,2019-02-05,311.479700,56.709038,High Risk,N
50,156800,2019-03-31,2019-01-01,296.936766,54.274785,High Risk,N
6,165027,2019-03-31,2019-01-01,287.699760,52.728659,High Risk,N


In [35]:
print('Value counts per indicator\n', df_q1_2019_weights3.loc[df_q1_2019_weights3['Bin']=='High Risk']
                                                    .sort_values(by='Risk_Score', ascending=False)['IND'].value_counts())
print('\nMean per indicator', df_q1_2019_weights3.loc[df_q1_2019_weights3['Bin']=='High Risk']
                                                      .groupby(['IND'])['Risk_Score'].mean())

Value counts per indicator
 N    5
Y    4
Name: IND, dtype: int64

Mean per indicator IND
N    63.234521
Y    71.144029
Name: Risk_Score, dtype: float64


### 1.3.4 Weighting System 4 Q1 2019
Probably classifies too many of the 'Y' in the Medium Risk bin, but that can be fixed in the next portion of this notebook. Also, good seperation of score between 'Y' and 'N'

In [36]:
df_q1_2019_weights4['IND'] = np.where(df_q1_2019_weights4['PRSCRB_PROV_LOC_ID'].isin(y_ind), 'Y', 'N')
df_q1_2019_weights4.loc[df_q1_2019_weights4['IND']=='Y']

,PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,Sum,Risk_Score,Bin,IND
32,122949,2019-03-31,2019-01-01,1150.964266,49.203111,Medium Risk,Y
54,107410,2019-03-31,2019-01-01,1859.170322,77.041835,High Risk,Y
3294,139433,2019-03-31,2019-01-01,1471.907659,61.819008,High Risk,Y
5051,16638,2019-03-31,2019-01-02,2443.216916,100.000000,High Risk,Y
5065,17522,2019-03-30,2019-01-01,1531.075542,64.144826,High Risk,Y
6751,153504,2019-03-31,2019-01-01,986.331844,42.731610,Medium Risk,Y
6803,4457,2019-03-31,2019-01-01,1064.648073,45.810126,Medium Risk,Y
10128,149990,2019-03-31,2019-01-01,1007.949790,43.581385,Medium Risk,Y


In [37]:
df_q1_2019_weights4.loc[df_q1_2019_weights4['Bin']=='High Risk'].sort_values(by='Risk_Score', ascending=False)

,PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,Sum,Risk_Score,Bin,IND
5051,16638,2019-03-31,2019-01-02,2443.216916,100.000000,High Risk,Y
1602,152853,2019-01-07,2019-01-07,1897.602462,78.552556,High Risk,N
54,107410,2019-03-31,2019-01-01,1859.170322,77.041835,High Risk,Y
7312,121269,2019-03-26,2019-01-02,1835.227303,76.100664,High Risk,N
12010,149181,2019-03-29,2019-01-02,1679.930557,69.996136,High Risk,N
5065,17522,2019-03-30,2019-01-01,1531.075542,64.144826,High Risk,Y
3294,139433,2019-03-31,2019-01-01,1471.907659,61.819008,High Risk,Y
50,156800,2019-03-31,2019-01-01,1278.637540,54.221793,High Risk,N
6,165027,2019-03-31,2019-01-01,1261.934547,53.565219,High Risk,N
7970,112675,2019-02-05,2019-02-05,1195.577914,50.956820,High Risk,N


In [38]:
print('Value counts per indicator\n', df_q1_2019_weights4.loc[df_q1_2019_weights4['Bin']=='High Risk']
                                                      .sort_values(by='Risk_Score', ascending=False)['IND'].value_counts())
print('\nMean per indicator', df_q1_2019_weights4.loc[df_q1_2019_weights4['Bin']=='High Risk']
                                                      .groupby(['IND'])['Risk_Score'].mean())


Value counts per indicator
 N    7
Y    4
Name: IND, dtype: int64

Mean per indicator IND
N    61.947756
Y    75.751417
Name: Risk_Score, dtype: float64


### 1.3.5 Weighting System 1 Q2 2019
Probably classifies too many of the 'N' in the High Risk bin, but that can be fixed in the next portion of this notebook. Also, good seperation of score between 'Y' and 'N'

In [39]:
df_q2_2019_weights1['IND'] = np.where(df_q2_2019_weights1['PRSCRB_PROV_LOC_ID'].isin(y_ind), 'Y', 'N')
df_q2_2019_weights1.loc[df_q2_2019_weights1['IND']=='Y']

,PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,Sum,Risk_Score,Bin,IND
3254,107410,2019-06-30,2019-04-01,106.992179,90.119516,High Risk,Y
5454,122949,2019-06-29,2019-04-01,47.261018,42.984788,Medium Risk,Y
7189,139433,2019-06-30,2019-04-01,69.852403,60.811979,High Risk,Y
9392,153504,2019-06-30,2019-04-01,56.714046,50.444310,High Risk,Y
11315,149990,2019-06-30,2019-04-01,60.343772,53.308580,High Risk,Y
11779,4457,2019-06-29,2019-04-01,55.804235,49.726365,Medium Risk,Y
12242,16638,2019-06-30,2019-04-01,119.513155,100.000000,High Risk,Y
12846,17522,2019-06-30,2019-04-01,82.286263,70.623718,High Risk,Y


In [40]:
df_q2_2019_weights1.loc[df_q2_2019_weights1['Bin']=='High Risk'].sort_values(by='Risk_Score', ascending=False)

,PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,Sum,Risk_Score,Bin,IND
12242,16638,2019-06-30,2019-04-01,119.513155,100.000000,High Risk,Y
3254,107410,2019-06-30,2019-04-01,106.992179,90.119516,High Risk,Y
4111,110160,2019-06-30,2019-04-02,84.257572,72.179307,High Risk,N
12846,17522,2019-06-30,2019-04-01,82.286263,70.623718,High Risk,Y
9148,135733,2019-05-03,2019-05-03,79.778706,68.644969,High Risk,N
13418,2755,2019-06-30,2019-04-01,73.076555,63.356203,High Risk,N
5441,122907,2019-06-19,2019-04-10,71.054428,61.760514,High Risk,N
7189,139433,2019-06-30,2019-04-01,69.852403,60.811979,High Risk,Y
10243,165027,2019-06-30,2019-04-01,69.842428,60.804107,High Risk,N
12232,166304,2019-06-14,2019-04-02,68.049231,59.389069,High Risk,N


In [41]:
print('Value counts per indicator\n', df_q2_2019_weights1.loc[df_q2_2019_weights1['Bin']=='High Risk']
                                                      .sort_values(by='Risk_Score', ascending=False)['IND'].value_counts())
print('\nMean per indicator', df_q2_2019_weights1.loc[df_q2_2019_weights1['Bin']=='High Risk']
                                                      .groupby(['IND'])['Risk_Score'].mean())


Value counts per indicator
 N    12
Y     6
Name: IND, dtype: int64

Mean per indicator IND
N    59.363550
Y    70.884684
Name: Risk_Score, dtype: float64


### 1.3.6 Weighting System 2 Q2 2019
Not very good classification of 'Y' and 'N'. Also not as good seperation of mean per group.

In [42]:
df_q2_2019_weights2['IND'] = np.where(df_q2_2019_weights2['PRSCRB_PROV_LOC_ID'].isin(y_ind), 'Y', 'N')
df_q2_2019_weights2.loc[df_q2_2019_weights2['IND']=='Y']

,PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,Sum,Risk_Score,Bin,IND
3254,107410,2019-06-30,2019-04-01,283.590276,50.857369,High Risk,Y
5454,122949,2019-06-29,2019-04-01,223.345283,41.019221,Medium Risk,Y
7189,139433,2019-06-30,2019-04-01,333.417042,58.994196,High Risk,Y
9392,153504,2019-06-30,2019-04-01,271.999468,48.964563,Medium Risk,Y
11315,149990,2019-06-30,2019-04-01,287.044675,51.421480,High Risk,Y
11779,4457,2019-06-29,2019-04-01,269.865095,48.616014,Medium Risk,Y
12242,16638,2019-06-30,2019-04-01,584.520647,100.000000,High Risk,Y
12846,17522,2019-06-30,2019-04-01,393.810046,68.856514,High Risk,Y


In [43]:
df_q2_2019_weights2.loc[df_q2_2019_weights2['Bin']=='High Risk'].sort_values(by='Risk_Score', ascending=False)

,PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,Sum,Risk_Score,Bin,IND
12242,16638,2019-06-30,2019-04-01,584.520647,100.000000,High Risk,Y
12846,17522,2019-06-30,2019-04-01,393.810046,68.856514,High Risk,Y
10243,165027,2019-06-30,2019-04-01,335.617930,59.353606,High Risk,N
7189,139433,2019-06-30,2019-04-01,333.417042,58.994196,High Risk,Y
1930,111813,2019-06-30,2019-04-01,311.465869,55.409518,High Risk,N
9871,156800,2019-06-30,2019-04-01,299.101786,53.390434,High Risk,N
10091,164110,2019-06-21,2019-04-15,290.899417,52.050968,High Risk,N
11315,149990,2019-06-30,2019-04-01,287.044675,51.421480,High Risk,Y
5441,122907,2019-06-19,2019-04-10,284.460982,50.999557,High Risk,N
3254,107410,2019-06-30,2019-04-01,283.590276,50.857369,High Risk,Y


In [44]:
print('Value counts per indicator\n', df_q2_2019_weights2.loc[df_q2_2019_weights2['Bin']=='High Risk']
                                              .sort_values(by='Risk_Score', ascending=False)['IND'].value_counts())
print('\nMean per indicator', df_q2_2019_weights2.loc[df_q2_2019_weights2['Bin']=='High Risk']
                                              .groupby(['IND'])['Risk_Score'].mean())


Value counts per indicator
 Y    5
N    5
Name: IND, dtype: int64

Mean per indicator IND
N    54.240817
Y    66.025912
Name: Risk_Score, dtype: float64


### 1.3.7 Weighting System 3 Q2 2019
Not very good classification of 'Y' and 'N'. Also not as good seperation of mean per group.

In [45]:
df_q2_2019_weights3['IND'] = np.where(df_q2_2019_weights3['PRSCRB_PROV_LOC_ID'].isin(y_ind), 'Y', 'N')
df_q2_2019_weights3.loc[df_q2_2019_weights3['IND']=='Y']

,PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,Sum,Risk_Score,Bin,IND
3254,107410,2019-06-30,2019-04-01,283.590276,50.857369,High Risk,Y
5454,122949,2019-06-29,2019-04-01,223.345283,41.019221,Medium Risk,Y
7189,139433,2019-06-30,2019-04-01,333.417042,58.994196,High Risk,Y
9392,153504,2019-06-30,2019-04-01,271.999468,48.964563,Medium Risk,Y
11315,149990,2019-06-30,2019-04-01,287.044675,51.421480,High Risk,Y
11779,4457,2019-06-29,2019-04-01,269.865095,48.616014,Medium Risk,Y
12242,16638,2019-06-30,2019-04-01,584.520647,100.000000,High Risk,Y
12846,17522,2019-06-30,2019-04-01,393.810046,68.856514,High Risk,Y


In [46]:
df_q2_2019_weights3.loc[df_q2_2019_weights3['Bin']=='High Risk'].sort_values(by='Risk_Score', ascending=False)

,PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,Sum,Risk_Score,Bin,IND
12242,16638,2019-06-30,2019-04-01,584.520647,100.000000,High Risk,Y
12846,17522,2019-06-30,2019-04-01,393.810046,68.856514,High Risk,Y
10243,165027,2019-06-30,2019-04-01,335.617930,59.353606,High Risk,N
7189,139433,2019-06-30,2019-04-01,333.417042,58.994196,High Risk,Y
1930,111813,2019-06-30,2019-04-01,311.465869,55.409518,High Risk,N
9871,156800,2019-06-30,2019-04-01,299.101786,53.390434,High Risk,N
10091,164110,2019-06-21,2019-04-15,290.899417,52.050968,High Risk,N
11315,149990,2019-06-30,2019-04-01,287.044675,51.421480,High Risk,Y
5441,122907,2019-06-19,2019-04-10,284.460982,50.999557,High Risk,N
3254,107410,2019-06-30,2019-04-01,283.590276,50.857369,High Risk,Y


In [47]:
print('Value counts per indicator\n', df_q2_2019_weights3.loc[df_q2_2019_weights3['Bin']=='High Risk']
                                                      .sort_values(by='Risk_Score', ascending=False)['IND'].value_counts())
print('\nMean per indicator', df_q2_2019_weights3.loc[df_q2_2019_weights3['Bin']=='High Risk']
                                                      .groupby(['IND'])['Risk_Score'].mean())


Value counts per indicator
 Y    5
N    5
Name: IND, dtype: int64

Mean per indicator IND
N    54.240817
Y    66.025912
Name: Risk_Score, dtype: float64


### 1.3.8 Weighting System 4 Q2 2019
Probably classifies too many of the 'Y' in the Medium Risk bin, but that can be fixed in the next portion of this notebook. Also, good seperation of score between 'Y' and 'N'

In [48]:
df_q2_2019_weights4['IND'] = np.where(df_q2_2019_weights4['PRSCRB_PROV_LOC_ID'].isin(y_ind), 'Y', 'N')
df_q2_2019_weights4.loc[df_q2_2019_weights4['IND']=='Y']

,PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,Sum,Risk_Score,Bin,IND
3254,107410,2019-06-30,2019-04-01,2385.008230,95.817114,High Risk,Y
5454,122949,2019-06-29,2019-04-01,940.934053,40.190282,Medium Risk,Y
7189,139433,2019-06-30,2019-04-01,1420.390554,58.659310,High Risk,Y
9392,153504,2019-06-30,2019-04-01,1126.716264,47.346753,Medium Risk,Y
11315,149990,2019-06-30,2019-04-01,1205.859778,50.395422,High Risk,Y
11779,4457,2019-06-29,2019-04-01,1127.232019,47.366621,Medium Risk,Y
12242,16638,2019-06-30,2019-04-01,2493.596062,100.000000,High Risk,Y
12846,17522,2019-06-30,2019-04-01,1679.481844,68.639703,High Risk,Y


In [49]:
df_q2_2019_weights4.loc[df_q2_2019_weights4['Bin']=='High Risk'].sort_values(by='Risk_Score', ascending=False)

,PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,Sum,Risk_Score,Bin,IND
12242,16638,2019-06-30,2019-04-01,2493.596062,100.000000,High Risk,Y
3254,107410,2019-06-30,2019-04-01,2385.008230,95.817114,High Risk,Y
4111,110160,2019-06-30,2019-04-02,1886.254490,76.604742,High Risk,N
12846,17522,2019-06-30,2019-04-01,1679.481844,68.639703,High Risk,Y
13418,2755,2019-06-30,2019-04-01,1601.765122,65.645996,High Risk,N
10243,165027,2019-06-30,2019-04-01,1473.575168,60.708021,High Risk,N
7189,139433,2019-06-30,2019-04-01,1420.390554,58.659310,High Risk,Y
1930,111813,2019-06-30,2019-04-01,1331.549331,55.237079,High Risk,N
9871,156800,2019-06-30,2019-04-01,1272.966356,52.980418,High Risk,N
11315,149990,2019-06-30,2019-04-01,1205.859778,50.395422,High Risk,Y


In [50]:
print('Value counts per indicator\n', df_q2_2019_weights4.loc[df_q2_2019_weights4['Bin']=='High Risk']
                                                      .sort_values(by='Risk_Score', ascending=False)['IND'].value_counts())
print('\nMean per indicator', df_q2_2019_weights4.loc[df_q2_2019_weights4['Bin']=='High Risk']
                                                      .groupby(['IND'])['Risk_Score'].mean())


Value counts per indicator
 Y    5
N    5
Name: IND, dtype: int64

Mean per indicator IND
N    62.235251
Y    74.702310
Name: Risk_Score, dtype: float64


### 1.3.9 Weight Scheme 4 has the best results
Based on fewest number of N to Y and best seperation of mean per indicator. Move forward with weight scheme 4 and tune threshold in the next section.

# 2.0 Find Optimal Threshold

## 2.1 Test Potential Risk Score Thresholds 
Use the same criteria as above to see which of the candidate thresholds is the best. The results need to be consistent in Q1 and Q2 for the new threshold to be determined as the best. The candidate thresholds used are 50, 55, 60, and 65.

### 2.1.1 Test on Q1 2019

In [51]:
candidates = [50, 55, 60, 65]

for i in candidates:
    print('Threshold: ', i)
    df_q1_2019_thresh1 = scoring_sum(df_for_scoring1, dictionary_of_weights_4, threshold=i)
    q1_2019_thresh1 = bins_for_scoring_groups(df_q1_2019_thresh1)

    df_q1_2019_thresh1['IND'] = np.where(df_q1_2019_thresh1['PRSCRB_PROV_LOC_ID'].isin(y_ind), 'Y', 'N')

    print('Value counts per indicator\n', df_q1_2019_thresh1.loc[df_q1_2019_thresh1['Bin']=='High Risk']
                                                      .sort_values(by='Risk_Score', ascending=False)['IND'].value_counts())
    print('\nMean per indicator', df_q1_2019_thresh1.loc[df_q1_2019_thresh1['Bin']=='High Risk']
                                                      .groupby(['IND'])['Risk_Score'].mean())
    print('\n')
    print('-----------------------')

Threshold:  50
Bin
Low Risk       13426
Medium Risk       14
High Risk         11
dtype: int64
List of suspected bad actors:  [165027, 156800, 107410, 152853, 139433, 16638, 17522, 121269, 111598, 112675, 149181]
Value counts per indicator
 N    7
Y    4
Name: IND, dtype: int64

Mean per indicator IND
N    61.947756
Y    75.751417
Name: Risk_Score, dtype: float64


-----------------------
Threshold:  55
Bin
Low Risk       13426
Medium Risk       18
High Risk          7
dtype: int64
List of suspected bad actors:  [107410, 152853, 139433, 16638, 17522, 121269, 149181]
Value counts per indicator
 Y    4
N    3
Name: IND, dtype: int64

Mean per indicator IND
N    74.883118
Y    75.751417
Name: Risk_Score, dtype: float64


-----------------------
Threshold:  60
Bin
Low Risk       13426
Medium Risk       18
High Risk          7
dtype: int64
List of suspected bad actors:  [107410, 152853, 139433, 16638, 17522, 121269, 149181]
Value counts per indicator
 Y    4
N    3
Name: IND, dtype: int64



### 2.1.2 Test on Q2 2019

In [52]:
candidates = [50, 55, 60, 65]

for i in candidates:
    print('Threshold: ', i)
    df_q2_2019_thresh1 = scoring_sum(df_for_scoring2, dictionary_of_weights_4, threshold=i)
    q2_2019_thresh1 = bins_for_scoring_groups(df_q2_2019_thresh1)

    df_q2_2019_thresh1['IND'] = np.where(df_q2_2019_thresh1['PRSCRB_PROV_LOC_ID'].isin(y_ind), 'Y', 'N')

    print('Value counts per indicator\n', df_q2_2019_thresh1.loc[df_q2_2019_thresh1['Bin']=='High Risk']
                                                      .sort_values(by='Risk_Score', ascending=False)['IND'].value_counts())
    print('\nMean per indicator', df_q2_2019_thresh1.loc[df_q2_2019_thresh1['Bin']=='High Risk']
                                                      .groupby(['IND'])['Risk_Score'].mean())
    print('\n')
    print('-----------------------')

Threshold:  50
Bin
Low Risk       13426
Medium Risk       13
High Risk         10
dtype: int64
List of suspected bad actors:  [111813, 107410, 110160, 139433, 156800, 165027, 149990, 16638, 17522, 2755]
Value counts per indicator
 Y    5
N    5
Name: IND, dtype: int64

Mean per indicator IND
N    62.235251
Y    74.702310
Name: Risk_Score, dtype: float64


-----------------------
Threshold:  55
Bin
Low Risk       13426
Medium Risk       15
High Risk          8
dtype: int64
List of suspected bad actors:  [111813, 107410, 110160, 139433, 165027, 16638, 17522, 2755]
Value counts per indicator
 Y    4
N    4
Name: IND, dtype: int64

Mean per indicator IND
N    64.548959
Y    80.779032
Name: Risk_Score, dtype: float64


-----------------------
Threshold:  60
Bin
Low Risk       13426
Medium Risk       17
High Risk          6
dtype: int64
List of suspected bad actors:  [107410, 110160, 165027, 16638, 17522, 2755]
Value counts per indicator
 Y    3
N    3
Name: IND, dtype: int64

Mean per indic

## 2.2 Use 55 as Risk Score Threshold
55 gives the best threshold because it filters out false positives while keeping true positives.

# 3.0 Apply Weights and Threshold to All Quarters
Now apply the optimal weight and threshold to the rest of the data. Then save the resultant risk scores as a csv in Cloud Pak for Data to be used in clustering.

## 3.1 Use Sort Order Weights and New Threshold on Q3 2019

In [53]:
# read in 1 quarter of data to start the scoring
# the '/project_data/data_asset' folder contains the quarterly data

df_q3_2019 = pd.read_csv('/project_data/data_asset/final_op_analysis_07012019_09302019.csv', usecols=usecols, dtype=dtype)
display(df_q3_2019.head())
print(df_q3_2019.shape)

,Max Dispense Date,Min Dispense Date,PRSCRB_PROV_LOC_ID,PR2,PR1,PR4,PR3,PR5,PR6,PR7,PR8,PR9,PR10,PR11,PR13,PR15,PR18,PR19,PR21,PR22,PR23,PR24,PR25,PR26,PR27,PR28,PR29,PR30
0,2019-08-26,2019-07-23,10531,1.0,11.0,0.057143,0.628571,0.333333,0.0,0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0.0,10.579864,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0
1,2019-07-12,2019-07-12,105316,1.0,60.0,1.000000,60.000000,0.052632,0.0,0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0.0,6.730443,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2019-09-19,2019-07-11,105327,1.5,65.0,0.126761,5.492958,0.333333,0.0,0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0.0,6.529557,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2019-09-06,2019-07-17,105329,1.0,16.0,0.038462,0.615385,0.035714,0.0,0,0.0,0.0,0.0,1.0,0.0,0,0,0.0,0.0,4.505377,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2019-09-20,2019-07-08,10533,3.0,90.0,0.080000,2.400000,0.545455,0.0,0,0.0,0.0,0.0,0.0,0.0,0,0,0.0,0.0,11.682132,0.0,0.0,0.0,0.0,0.0,0.5,0.0,0.0


(13286, 28)


In [54]:
# make list for preprocessing
features = [c for c in df_q3_2019.columns if c not in index+to_drop]
num_features = df_q3_2019[features].select_dtypes('number').columns.tolist()

In [55]:
# define the data preprocessor
preprocessor_refine, features_names = preprocessor_refined(binary_features, num_features)

# compute the scoring from the preprocessed data
df_for_scoring3 = apply_data_transform(preprocessor_refine, df_q3_2019, features_names, index)
df_for_scoring3.head()

,,,PR15,PR18,PR2,PR1,PR4,PR3,PR5,PR6,PR7,PR8,PR9,PR10,PR13,PR19,PR21,PR22,PR23,PR24,PR25,PR26,PR27,PR28,PR29,PR30
PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,,,,,,,,,,,,,,,,,,,,,,,,
10531,2019-08-26,2019-07-23,0,0,-0.681968,-0.692612,-0.638124,-0.392146,0.890790,-0.266907,-0.175096,-0.114718,-0.037663,-0.259898,-0.564756,-0.033668,-0.236864,-0.306524,-0.217989,-0.075769,-0.348536,-0.046347,-0.012668,0.978617,-0.218898,-0.076756
105316,2019-07-12,2019-07-12,0,0,-0.681968,-0.114968,0.935275,1.010953,-0.484800,-0.266907,-0.175096,-0.114718,-0.037663,-0.259898,-0.564756,-0.033668,-0.236864,-0.414217,-0.217989,-0.075769,-0.348536,-0.046347,-0.012668,-0.760112,-0.218898,-0.076756
105327,2019-09-19,2019-07-11,0,0,-0.138193,-0.056025,-0.521949,-0.277188,0.890790,-0.266907,-0.175096,-0.114718,-0.037663,-0.259898,-0.564756,-0.033668,-0.236864,-0.419837,-0.217989,-0.075769,-0.348536,-0.046347,-0.012668,-0.760112,-0.218898,-0.076756
105329,2019-09-06,2019-07-17,0,0,-0.681968,-0.633669,-0.669299,-0.392458,-0.567704,-0.266907,-0.175096,-0.114718,-0.037663,-0.259898,-0.564756,-0.033668,-0.236864,-0.476467,-0.217989,-0.075769,-0.348536,-0.046347,-0.012668,-0.760112,-0.218898,-0.076756
10533,2019-09-20,2019-07-08,0,0,1.493135,0.238691,-0.599981,-0.350283,1.930298,-0.266907,-0.175096,-0.114718,-0.037663,-0.259898,-0.564756,-0.033668,-0.236864,-0.275686,-0.217989,-0.075769,-0.348536,-0.046347,-0.012668,0.978617,-0.218898,-0.076756


In [58]:
# using sort order weights

#Scale the scoring
df_q3_2019_weights = scoring_sum(df_for_scoring3, dictionary_of_weights_4, threshold=55)

# output the bins with sizes
q3_2019_weights = bins_for_scoring_groups(df_q3_2019_weights)

Bin
Low Risk       13265
Medium Risk       15
High Risk          6
dtype: int64
List of suspected bad actors:  [107410, 148485, 139433, 165027, 16638, 17522]


## 3.2 Use Sort Order Weights and New Threshold on Q4 2019

In [59]:
# read in 1 quarter of data to start the scoring
# the '/project_data/data_asset' folder contains the quarterly data

df_q4_2019 = pd.read_csv('/project_data/data_asset/final_op_analysis_10012019_12312019.csv', usecols=usecols, dtype=dtype)
display(df_q4_2019.head())
print(df_q4_2019.shape)

,Max Dispense Date,Min Dispense Date,PRSCRB_PROV_LOC_ID,PR2,PR1,PR4,PR3,PR5,PR6,PR7,PR8,PR9,PR10,PR11,PR13,PR15,PR18,PR19,PR21,PR22,PR23,PR24,PR25,PR26,PR27,PR28,PR29,PR30
0,2019-10-29,2019-10-29,103223,1.000,20.000000,1.000000,20.000000,0.000000,0.000,0,0.0,0.0,0.0,1.000000,0.00,0,0,0.0,0.000000,3.910989,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0
1,2019-12-28,2019-10-04,103229,1.000,14.333333,0.034884,0.500000,0.142857,0.000,0,0.0,0.0,0.0,0.666667,0.00,0,0,0.0,0.000000,38.711733,0.0,0.0,0.0,0.0,0.0,1.00,0.000000,0.0
2,2019-12-23,2019-10-29,103230,1.000,18.333333,0.053571,0.982143,0.142857,0.000,0,0.0,0.0,0.0,1.000000,0.00,0,0,0.0,0.333333,77.630627,0.0,0.0,0.0,0.0,0.0,0.00,0.000000,0.0
3,2019-12-31,2019-10-10,103231,2.875,130.500000,0.277108,12.578313,0.084906,0.125,1,0.0,0.0,0.0,0.521739,0.53,0,0,0.0,0.000000,5.152391,0.0,0.0,0.0,0.0,0.0,0.25,0.000000,0.0
4,2019-12-27,2019-10-23,103248,1.500,10.500000,0.045455,0.318182,0.000000,0.000,0,0.0,0.0,0.5,0.333333,0.00,0,0,0.0,0.000000,5.682214,0.0,0.0,0.0,0.0,0.0,0.00,0.333333,0.0


(13389, 28)


In [60]:
# make list for preprocessing
features = [c for c in df_q4_2019.columns if c not in index+to_drop]
num_features = df_q4_2019[features].select_dtypes('number').columns.tolist()

In [61]:
# define the data preprocessor
preprocessor_refine, features_names = preprocessor_refined(binary_features, num_features)

# compute the scoring from the preprocessed data
df_for_scoring4 = apply_data_transform(preprocessor_refine, df_q4_2019, features_names, index)
df_for_scoring4.head()

,,,PR15,PR18,PR2,PR1,PR4,PR3,PR5,PR6,PR7,PR8,PR9,PR10,PR13,PR19,PR21,PR22,PR23,PR24,PR25,PR26,PR27,PR28,PR29,PR30
PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,,,,,,,,,,,,,,,,,,,,,,,,
103223,2019-10-29,2019-10-29,0,0,-0.677356,-0.589319,0.888926,0.055985,-0.740064,-0.259495,-0.177731,-0.116266,-0.035005,-0.248827,-0.55482,-0.054764,-0.233900,-0.330284,-0.207114,-0.070842,-0.352054,-0.06812,-0.012668,-0.741685,-0.364240,-0.06679
103229,2019-12-28,2019-10-04,0,0,-0.677356,-0.658133,-0.680604,-0.408234,-0.038078,-0.259495,-0.177731,-0.116266,-0.035005,-0.248827,-0.55482,-0.054764,-0.233900,0.292773,-0.207114,-0.070842,-0.352054,-0.06812,-0.012668,2.706273,-0.364240,-0.06679
103230,2019-12-23,2019-10-29,0,0,-0.677356,-0.609558,-0.650213,-0.396756,-0.038078,-0.259495,-0.177731,-0.116266,-0.035005,-0.248827,-0.55482,-0.054764,3.444975,0.989559,-0.207114,-0.070842,-0.352054,-0.06812,-0.012668,-0.741685,-0.364240,-0.06679
103231,2019-12-31,2019-10-10,0,0,1.396202,0.752558,-0.286683,-0.120697,-0.322846,0.483916,0.345842,-0.116266,-0.035005,-0.248827,2.43451,-0.054764,-0.233900,-0.308058,-0.207114,-0.070842,-0.352054,-0.06812,-0.012668,0.120304,-0.364240,-0.06679
103248,2019-12-27,2019-10-23,0,0,-0.124407,-0.704684,-0.663413,-0.412562,-0.740064,-0.259495,-0.177731,-0.116266,-0.035005,4.686694,-0.55482,-0.054764,-0.233900,-0.298573,-0.207114,-0.070842,-0.352054,-0.06812,-0.012668,-0.741685,2.273243,-0.06679


In [62]:
# using sort order weights

#Scale the scoring
df_q4_2019_weights = scoring_sum(df_for_scoring4, dictionary_of_weights_4, threshold=55)

# output the bins with sizes
q4_2019_weights = bins_for_scoring_groups(df_q4_2019_weights)

Bin
Low Risk       13370
Medium Risk       13
High Risk          6
dtype: int64
List of suspected bad actors:  [126953, 131365, 148485, 16638, 170094, 17522]


## 3.3 Use Sort Order Weights and New Threshold on Q1 2020

In [63]:
# read in 1 quarter of data to start the scoring
# the '/project_data/data_asset' folder contains the quarterly data

df_q1_2020 = pd.read_csv('/project_data/data_asset/final_op_analysis_01012020_03312020.csv', usecols=usecols, dtype=dtype)
display(df_q1_2020.head())
print(df_q1_2020.shape)

,Max Dispense Date,Min Dispense Date,PRSCRB_PROV_LOC_ID,PR2,PR1,PR4,PR3,PR5,PR6,PR7,PR8,PR9,PR10,PR11,PR13,PR15,PR18,PR19,PR21,PR22,PR23,PR24,PR25,PR26,PR27,PR28,PR29,PR30
0,2020-01-06,2020-01-06,103252,1.000000,20.000000,1.000000,20.000000,0.111111,0.000000,0,0.000000,0.0,0.000000,1.000000,0.00,0,0,0.0,0.000000,4.105901,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
1,2020-03-31,2020-01-07,103257,1.769231,56.846154,0.270588,8.694118,0.049107,0.000000,0,0.000000,0.0,0.000000,0.391304,0.60,0,0,0.0,0.000000,10.205747,0.0,0.000000,0.000000,0.000000,0.0,0.307692,0.043478,0.0
2,2020-03-31,2020-01-01,103278,4.254545,252.363636,5.142857,305.054945,0.172986,0.009091,4,0.001188,0.0,0.072727,0.529915,0.42,0,0,0.0,0.054545,14.011228,0.0,0.009091,0.136364,0.036364,0.0,0.581818,0.014957,0.0
3,2020-03-13,2020-01-02,103291,1.916667,119.583333,0.319444,19.930556,0.005405,0.000000,0,0.000000,0.0,0.166667,0.043478,0.51,0,0,0.0,0.000000,12.290849,0.0,0.000000,0.000000,0.000000,0.0,0.166667,0.130435,0.0
4,2020-01-03,2020-01-03,103293,1.000000,10.000000,1.000000,10.000000,0.111111,0.000000,0,0.000000,0.0,0.000000,1.000000,0.00,0,0,0.0,0.000000,5.457991,0.0,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0


(13239, 28)


In [64]:
# make list for preprocessing
features = [c for c in df_q1_2020.columns if c not in index+to_drop]
num_features = df_q1_2020[features].select_dtypes('number').columns.tolist()

In [66]:
# define the data preprocessor
preprocessor_refine, features_names = preprocessor_refined(binary_features, num_features)

# compute the scoring from the preprocessed data
df_for_scoring1_2020 = apply_data_transform(preprocessor_refine, df_q1_2020, features_names, index)
df_for_scoring1_2020.head()

,,,PR15,PR18,PR2,PR1,PR4,PR3,PR5,PR6,PR7,PR8,PR9,PR10,PR13,PR19,PR21,PR22,PR23,PR24,PR25,PR26,PR27,PR28,PR29,PR30
PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,,,,,,,,,,,,,,,,,,,,,,,,
103252,2020-01-06,2020-01-06,0,0,-0.710371,-0.603336,0.877724,0.059010,-0.151332,-0.261152,-0.180171,-0.112323,-0.04464,-0.245197,-0.559420,-0.050697,-0.224118,-0.190635,-0.202789,-0.075319,-0.343121,-0.068113,-0.015901,-0.729790,-0.255486,-0.077328
103257,2020-03-31,2020-01-07,0,0,0.148788,-0.152918,-0.302954,-0.214233,-0.461664,-0.261152,-0.180171,-0.112323,-0.04464,-0.245197,2.848296,-0.050697,-0.224118,-0.130599,-0.202789,-0.075319,-0.343121,-0.068113,-0.015901,0.367443,0.230096,-0.077328
103278,2020-03-31,2020-01-01,0,0,2.924651,2.237144,7.583647,6.948279,0.158354,-0.206802,1.928368,1.802789,-0.04464,0.514448,1.825981,-0.050697,0.400419,-0.093144,-0.202789,0.247885,0.640264,1.439753,-0.015901,1.344977,-0.088437,-0.077328
103291,2020-03-13,2020-01-02,0,0,0.313460,0.614000,-0.223872,0.057332,-0.680394,-0.261152,-0.180171,-0.112323,-0.04464,1.495658,2.337138,-0.050697,-0.224118,-0.110077,-0.202789,-0.075319,-0.343121,-0.068113,-0.015901,-0.135455,1.201260,-0.077328
103293,2020-01-03,2020-01-03,0,0,-0.710371,-0.725579,0.877724,-0.182672,-0.151332,-0.261152,-0.180171,-0.112323,-0.04464,-0.245197,-0.559420,-0.050697,-0.224118,-0.177327,-0.202789,-0.075319,-0.343121,-0.068113,-0.015901,-0.729790,-0.255486,-0.077328


In [67]:
# using sort order weights

#Scale the scoring
df_q1_2020_weights = scoring_sum(df_for_scoring1_2020, dictionary_of_weights_4, threshold=55)

# output the bins with sizes
q1_2020_weights = bins_for_scoring_groups(df_q1_2020_weights)

Bin
Low Risk       13223
Medium Risk        8
High Risk          8
dtype: int64
List of suspected bad actors:  [101951, 126953, 139433, 148485, 170094, 16638, 17522, 175614]


## 3.4 Use Sort Order Weights and New Threshold on Q2 2020

In [68]:
# read in 1 quarter of data to start the scoring
# the '/project_data/data_asset' folder contains the quarterly data

df_q2_2020 = pd.read_csv('/project_data/data_asset/final_op_analysis_04012020_06302020.csv', usecols=usecols, dtype=dtype)
display(df_q2_2020.head())
print(df_q2_2020.shape)

,Max Dispense Date,Min Dispense Date,PRSCRB_PROV_LOC_ID,PR2,PR1,PR4,PR3,PR5,PR6,PR7,PR8,PR9,PR10,PR11,PR13,PR15,PR18,PR19,PR21,PR22,PR23,PR24,PR25,PR26,PR27,PR28,PR29,PR30
0,2020-05-28,2020-04-30,103451,1.666667,36.333333,0.172414,3.758621,0.230769,0.333333,0,0.000000,0.0,0.000,0.000000,0.00,0,0,0.0,0.000000,8.275003,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0
1,2020-06-28,2020-04-01,103452,2.958333,107.083333,0.797753,28.876404,0.048426,0.000000,0,0.000834,0.0,0.125,0.225352,0.19,0,0,0.0,0.166667,10.717675,0.0,0.0,0.083333,0.0,0.0,0.041667,0.098592,0.0
2,2020-06-24,2020-05-19,103453,1.500000,33.000000,0.162162,3.567568,0.294118,0.000000,0,0.000000,0.0,0.000,0.166667,0.00,0,0,0.0,0.000000,46.153627,0.0,0.0,0.250000,0.0,0.0,0.000000,0.000000,0.0
3,2020-06-29,2020-04-06,103458,1.333333,59.833333,0.094118,4.223529,0.049505,0.000000,0,0.000000,0.0,0.000,0.250000,0.00,0,0,0.0,0.000000,2.264750,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0
4,2020-05-21,2020-05-21,103463,1.000000,10.000000,1.000000,10.000000,0.083333,0.000000,0,0.000000,0.0,0.000,0.000000,0.00,0,0,0.0,0.000000,2.235063,0.0,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.0


(12563, 28)


In [69]:
# make list for preprocessing
features = [c for c in df_q2_2020.columns if c not in index+to_drop]
num_features = df_q2_2020[features].select_dtypes('number').columns.tolist()

In [70]:
# define the data preprocessor
preprocessor_refine, features_names = preprocessor_refined(binary_features, num_features)

# compute the scoring from the preprocessed data
df_for_scoring2_2020 = apply_data_transform(preprocessor_refine, df_q2_2020, features_names, index)
df_for_scoring2_2020.head()

,,,PR15,PR18,PR2,PR1,PR4,PR3,PR5,PR6,PR7,PR8,PR9,PR10,PR13,PR19,PR21,PR22,PR23,PR24,PR25,PR26,PR27,PR28,PR29,PR30
PRSCRB_PROV_LOC_ID,Max Dispense Date,Min Dispense Date,,,,,,,,,,,,,,,,,,,,,,,,
103451,2020-05-28,2020-04-30,0,0,0.036755,-0.404109,-0.448845,-0.329019,0.399970,1.769841,-0.176465,-0.112155,-0.03679,-0.248761,-0.548529,-0.046667,-0.240754,-0.151281,-0.19523,-0.072993,-0.351752,-0.065766,-0.015662,-0.706418,-0.203609,-0.06903
103452,2020-06-28,2020-04-01,0,0,1.462783,0.432119,0.571643,0.277220,-0.500768,-0.249041,-0.176465,1.241387,-0.03679,1.044907,0.563459,-0.046667,1.683518,-0.126756,-0.19523,-0.072993,0.235899,-0.065766,-0.015662,-0.557586,1.074101,-0.06903
103453,2020-06-24,2020-05-19,0,0,-0.147249,-0.443507,-0.465574,-0.333630,0.712899,-0.249041,-0.176465,-0.112155,-0.03679,-0.248761,-0.548529,-0.046667,-0.240754,0.229029,-0.19523,-0.072993,1.411202,-0.065766,-0.015662,-0.706418,-0.203609,-0.06903
103458,2020-06-29,2020-04-06,0,0,-0.331252,-0.126352,-0.576616,-0.317798,-0.495439,-0.249041,-0.176465,-0.112155,-0.03679,-0.248761,-0.548529,-0.046667,-0.240754,-0.211625,-0.19523,-0.072993,-0.351752,-0.065766,-0.015662,-0.706418,-0.203609,-0.06903
103463,2020-05-21,2020-05-21,0,0,-0.699259,-0.715355,0.901689,-0.178378,-0.328334,-0.249041,-0.176465,-0.112155,-0.03679,-0.248761,-0.548529,-0.046667,-0.240754,-0.211923,-0.19523,-0.072993,-0.351752,-0.065766,-0.015662,-0.706418,-0.203609,-0.06903


In [71]:
# using sort order weights

#Scale the scoring
df_q2_2020_weights = scoring_sum(df_for_scoring2_2020, dictionary_of_weights_4, threshold=55)

# output the bins with sizes
q2_2020_weights = bins_for_scoring_groups(df_q2_2020_weights)

Bin
Low Risk       12536
Medium Risk       21
High Risk          6
dtype: int64
List of suspected bad actors:  [126953, 120451, 170094, 17522, 175614, 16638]


## 3.5 Write risk score to csv for clustering and dashboard

In [127]:
# overwrite csv files for clustering


# df_q1_2019_weights4[['PRSCRB_PROV_LOC_ID', 'Risk_Score']].to_csv('/project_data/data_asset/2019q1_riskscore.csv', index=False) 
# df_q2_2019_weights4[['PRSCRB_PROV_LOC_ID', 'Risk_Score']].to_csv('/project_data/data_asset/2019q2_riskscore.csv', index=False) 
# df_q3_2019_weights[['PRSCRB_PROV_LOC_ID', 'Risk_Score']].to_csv('/project_data/data_asset/2019q3_riskscore.csv', index=False) 
# df_q4_2019_weights[['PRSCRB_PROV_LOC_ID', 'Risk_Score']].to_csv('/project_data/data_asset/2019q4_riskscore.csv', index=False) 
# df_q1_2020_weights[['PRSCRB_PROV_LOC_ID', 'Risk_Score']].to_csv('/project_data/data_asset/2020q1_riskscore.csv', index=False) 
# df_q2_2020_weights[['PRSCRB_PROV_LOC_ID', 'Risk_Score']].to_csv('/project_data/data_asset/2020q2_riskscore.csv', index=False) 



### 3.6 Get extract for validation
For the purpose of validating this approach, extract some metadata about suspicious providers and get feedback about whether or not they are truly suspicious.